Q10: Data Wrangling

Data wrangling on real estate prices.

Use RealEstate_Prices.csv from the same folder. This is the complete Housing Prices dataset with 545 rows and 13 columns, matching the supplied practical.

Source: https://www.kaggle.com/datasets/yasserh/housing-prices-dataset

The downloaded data is unchanged and saved as RealEstate_Prices.csv. Cleaned data is saved separately as cleaned_realestate_prices.csv.

No related location or amenities dataset with matching identifiers is available, so the conditional merge step is not applicable. The large furnished subset is shown separately. Outlier handling caps price and area; bedroom, bathroom and story counts are preserved.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("RealEstate_Prices.csv")
print(df.head())
df.info()
print("\nMissing Values:")
print(df.isnull().sum())

      price  area  bedrooms  ...  parking  prefarea furnishingstatus
0  13300000  7420         4  ...        2       yes        furnished
1  12250000  8960         4  ...        3        no        furnished
2  12250000  9960         3  ...        2       yes   semi-furnished
3  12215000  7500         4  ...        3       yes        furnished
4  11410000  7420         4  ...        2        no        furnished

[5 rows x 13 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    objec

In [2]:
# Clean column names
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^a-zA-Z0-9_]", "", regex=True)
)

In [3]:
# Remove duplicates, extra spaces and missing values
df = df.drop_duplicates().copy()

for col in df.select_dtypes("object"):
    df[col] = df[col].str.strip()

df = df.dropna()

In [4]:
# Identify and cap outliers in continuous measurements
for col in ["price", "area"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    print(col, "outliers:", ((df[col] < lower) | (df[col] > upper)).sum())
    df[col] = df[col].clip(lower, upper)

price outliers: 15
area outliers: 12


In [5]:
# Select large furnished properties
large_furnished = df.loc[(df["area"] >= 5000) & (df["furnishingstatus"] == "furnished")].copy()
print("Large Furnished Properties:")
print(large_furnished.head())
print("Subset Shape:", large_furnished.shape)

Large Furnished Properties:
     price  area  bedrooms  ...  parking  prefarea furnishingstatus
0  9205000  7420         4  ...        2       yes        furnished
1  9205000  8960         4  ...        3        no        furnished
3  9205000  7500         4  ...        3       yes        furnished
4  9205000  7420         4  ...        2        no        furnished
8  9205000  8100         4  ...        2       yes        furnished

[5 rows x 13 columns]
Subset Shape: (76, 13)


In [6]:
# Average prices by furnishing status
summary = df.groupby("furnishingstatus")["price"].agg(["count", "mean", "median"])
print(summary.round(2))

                  count        mean     median
furnishingstatus                              
furnished           140  5398196.00  5075000.0
semi-furnished      227  4879986.78  4585000.0
unfurnished         178  4005179.78  3430000.0


In [7]:
# Encode categorical variables
categorical = df.select_dtypes("object").columns
df = pd.get_dummies(df, columns=categorical, drop_first=True)
print("Cleaned Data:")
print(df.head())
print("Cleaned Shape:", df.shape)

Cleaned Data:
     price  area  ...  furnishingstatus_semi-furnished  furnishingstatus_unfurnished
0  9205000  7420  ...                            False                         False
1  9205000  8960  ...                            False                         False
2  9205000  9960  ...                             True                         False
3  9205000  7500  ...                            False                         False
4  9205000  7420  ...                            False                         False

[5 rows x 14 columns]
Cleaned Shape: (545, 14)


In [8]:
# Save the cleaned dataset
df.to_csv("cleaned_realestate_prices.csv", index=False)
print("Saved cleaned_realestate_prices.csv")

Saved cleaned_realestate_prices.csv
